<a href="https://colab.research.google.com/github/SakshamPainuly/underwater-creature-tracking/blob/anu%2Fmodel-development/my_model/colab_train_rtdetr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWCT — Colab RT-DETR training (E003, YOLO vs RT-DETR comparison)

Same pipeline as `notebooks/colab_train_baseline.ipynb` (YOLO E001), run for **E003**
(`rtdetr-l.pt`, D-015). Runbook: `docs/COLAB.md`. This notebook only calls repository scripts.

- Runtime: **GPU** (Runtime → Change runtime type). Never run training on the local PC.
- Same frozen dataset **v1** view as E001 (YOLO-format labels, no conversion needed);
  zExtra never reaches training; the training data file has **no test split**.
- `train_yolo.py` detects the RT-DETR checkpoint and uses Ultralytics' `RTDETR` class.
- Runs, checkpoints and records go to `MyDrive/uwct/runs/E003*/` (outside Git).
- Before the full run: the smoke test (cell 12) must finish **without NaN losses** and within GPU
  memory at batch 16. If not, stop — changing batch or AMP needs a new decision (D-015).
- The **FINAL TEST EVALUATION** section runs only after E003 is registered, reviewed and declared
  in `experiments/final_candidates.csv`.

In [1]:
# 1. Settings -- set COMMIT (full 40-character sha of the commit to run); keep everything else.
import os, re, subprocess, sys

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/model-development"
COMMIT = "8989077f595c3d14c04bb35d1cba7410b3fa3f0c"  # REQUIRED: `git rev-parse HEAD` on the local PC, after that commit is pushed
EXPERIMENT_CONFIG = "configs/experiments/E003_rtdetr_baseline.yaml"
EXPERIMENT_ID = "E003"
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"  # Colab Secrets: fine-grained, this repo only, Contents read-only

REPO_DIR = "/content/uwct"                             # fixed: resumed runs expect the same dataset path
DRIVE_RAW = "/content/drive/MyDrive/uwct/raw/biology"  # Set 100 + Set 2000 only (no zExtra)
RUNS_ROOT = "/content/drive/MyDrive/uwct/runs"         # persistent run folders
LOCAL_RAW = "/content/raw/biology"                     # fast local copy used for training
WEIGHTS_DIR = "/content/weights"                       # pretrained checkpoints downloaded by Ultralytics

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")

In [2]:
# 2. GPU present?
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU -- switch the runtime to GPU")

Tue Oct 13 15:43:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 3. Mount Google Drive (raw training subset + persistent runs folder)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS_ROOT, exist_ok=True)

Mounted at /content/drive


In [4]:

# 4. Check out the exact commit (read-only token from Colab Secrets; see docs/COLAB.md).
# Git obtains credentials through GIT_ASKPASS.
# The token is not placed in the Git URL, command-line arguments, or helper file.

import os
import stat
import subprocess
from google.colab import userdata

REPO_URL = f"https://github.com/{REPO}.git"
ASKPASS = "/content/uwct_git_askpass.sh"

with open(ASKPASS, "w") as f:
    f.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  Username*) echo "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
        'esac\n'
    )

os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    """Local Git commands; no credentials involved."""
    r = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    """Run clone/fetch with the Colab Secret available to this Git process."""

    # Use the exact name of your Colab Secret.
    token = userdata.get("uwct_token")

    if not token:
        raise RuntimeError(
            'Colab Secret "uwct_token" is missing, empty, '
            "or notebook access is disabled."
        )

    env = {
        **os.environ,
        "GIT_ASKPASS": ASKPASS,
        "GIT_TERMINAL_PROMPT": "0",
        "UWCT_GIT_TOKEN": token,
    }

    try:
        r = subprocess.run(
            ["git", "-c", "credential.helper=", *args],
            cwd=cwd,
            env=env,
            capture_output=True,
            text=True
        )

        err = r.stderr.replace(token, "***")

        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = (
            os.path.isfile(config)
            and token in open(config, encoding="utf-8").read()
        )

    finally:
        env.pop("UWCT_GIT_TOKEN", None)
        token = None

    if r.returncode != 0:
        if "403" in err or "not granted" in err:
            err += (
                '\nThe token in Colab Secret "uwct_token" may not have '
                f"read access to repository {REPO}. Check the token's "
                "resource owner, repository access, and Contents: Read-only "
                "permission. See docs/COLAB.md, 'One-time setup'."
            )

        raise RuntimeError(f"git {args[0]} failed:\n{err}")

    assert not leaked, "Token found in .git/config"


# Clone the repository or fetch the requested branch.
if not os.path.isdir(REPO_DIR):
    git_remote(
        "clone",
        "--branch", BRANCH,
        REPO_URL,
        REPO_DIR,
        cwd="/content"
    )
else:
    git_remote("fetch", "origin", BRANCH)

# Check out and verify the exact commit.
git("checkout", "--detach", COMMIT)

assert git("rev-parse", "HEAD") == COMMIT
assert git("remote", "get-url", "origin") == REPO_URL

print(
    "commit:", COMMIT,
    "| working tree:", git("status", "--porcelain") or "clean"
)


commit: 8989077f595c3d14c04bb35d1cba7410b3fa3f0c | working tree: clean


In [5]:
# 5. Install the pinned GPU requirements (ultralytics==8.4.155; torch/CUDA come from Colab)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, "
   "'| torch', torch.__version__, '| cuda', torch.version.cuda)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 7.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ultralytics 8.4.155 | torch 2.11.0+cu130 | cuda 13.0


In [6]:
# 6. Environment report for this session (each run record stores its own copy too)
sh(sys.executable, "scripts/env_report.py", "--out", "/content/env_report_session.json")

{
  "timestamp_utc": "2026-10-13T15:43:54+00:00",
  "os": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "python": "3.13.15",
  "cpu_count": 2,
  "torch": "2.11.0+cu130",
  "cuda_available": true,
  "cuda_version": "13.0",
  "cudnn_version": 92700,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "vram_gb": 14.56
    }
  ],
  "ultralytics": "8.4.155",
  "nvidia_smi": [
    {
      "name": "Tesla T4",
      "memory_total": "15360 MiB",
      "driver_version": "580.82.07"
    }
  ]
}


In [7]:
# 7. Reconstruct Biology raw data locally from the shared ZIP.
#    Nothing is written back to Google Drive.

import os
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

ZIP_ID = "1_97E8IGX_euJLAzXlyfpukhvRjg2XhfM"
ZIP_PATH = Path("/content/biology_dataset.zip")
LOCAL_RAW = Path("/content/raw/biology")

# Start clean
if LOCAL_RAW.exists():
    shutil.rmtree(LOCAL_RAW)

LOCAL_RAW.mkdir(parents=True, exist_ok=True)

# Install gdown
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True
)

# Download shared ZIP
print("Downloading Biology ZIP...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "gdown",
        f"https://drive.google.com/uc?id={ZIP_ID}",
        "-O",
        str(ZIP_PATH),
    ],
    check=True
)

print("Downloaded:", ZIP_PATH)
print("Size:", ZIP_PATH.stat().st_size, "bytes")

# Extract outer ZIP
OUTER = Path("/content/biology_outer")
if OUTER.exists():
    shutil.rmtree(OUTER)
OUTER.mkdir()

print("\nExtracting outer ZIP...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUTER)

ROOT = OUTER / "AI Lobster Training Dataset"

# Set 100
print("\nCopying Set 100...")
shutil.copytree(
    ROOT / "Set 100",
    LOCAL_RAW / "Set 100"
)

# Set 2000 images
IMG_ZIP = ROOT / "Set 2000" / "batch_2 (2k - imgOnly).zip"
IMG_DST = LOCAL_RAW / "Set 2000" / "batch_2 (2k - imgOnly)"

print("Extracting Set 2000 images...")
IMG_DST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(IMG_ZIP, "r") as z:
    z.extractall(IMG_DST)

# Set 2000 labels
LABEL_ZIP = ROOT / "Set 2000" / "batch_2_2k_labels.zip"
LABEL_DST = LOCAL_RAW / "Set 2000" / "batch_2_2k_labels"

print("Extracting Set 2000 labels...")
LABEL_DST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LABEL_ZIP, "r") as z:
    z.extractall(LABEL_DST)

# zExtra stays in raw data for fingerprint verification,
# but remains excluded from the frozen training view.
print("Copying zExtra Photos...")
shutil.copytree(
    ROOT / "zExtra Photos",
    LOCAL_RAW / "zExtra Photos"
)

# Project documents
for name in [
    "Lobster AI Detection Training Data.docx",
    "What's been done.xlsx",
]:
    src = ROOT / name
    if src.exists():
        shutil.copy2(src, LOCAL_RAW / name)

# Tell all subsequent project scripts where raw data lives
os.environ["UWCT_RAW_ROOT"] = str(LOCAL_RAW)

print("\n=== RAW ROOT ===")
print(LOCAL_RAW)

print("\n=== TOP LEVEL ===")
for p in sorted(LOCAL_RAW.iterdir()):
    print(p.name)

assert (LOCAL_RAW / "Set 100").is_dir()
assert (LOCAL_RAW / "Set 2000").is_dir()
assert (LOCAL_RAW / "zExtra Photos").is_dir()

print("\nCell 7 reconstruction complete.")

Downloaded: /content/biology_dataset.zip
Size: 5478326324 bytes

Extracting outer ZIP...

Copying Set 100...
Extracting Set 2000 images...
Extracting Set 2000 labels...
Copying zExtra Photos...

=== RAW ROOT ===
/content/raw/biology

=== TOP LEVEL ===
Set 100
Set 2000
What's been done.xlsx
zExtra Photos

Cell 7 reconstruction complete.


In [8]:
# 8. Raw-data check of the local copy (structural: paths + sizes; txt contents hashed). Must MATCH.
sh(sys.executable, "scripts/check_raw_unchanged.py", "--root", LOCAL_RAW, "--subset", "training")

check: structural (paths+sizes; txt/xlsx/docx hashed); subset: training; root: /content/raw/biology
fingerprint: /content/uwct/data/processed/raw_fingerprint_d011_post_cleanup.json
files expected: 3969  present: 3969
  missing: 0
  added: 0
  size changed: 0
  content changed: 0
RAW DATA MATCHES FINGERPRINT


In [9]:
# 9. Build the YOLO view of frozen v1 (verifies FROZEN.json hashes first; symlinks to the local copy)
sh(sys.executable, "scripts/prepare_yolo_dataset.py", "--version", "v1")

raw data root: /content/raw/biology
{
  "dataset_name": "lobster_v1",
  "dataset_version": "v1",
  "version_status": "frozen",
  "dataset_fingerprint": "9f2bf8573df4354e1373786967a1155802e4cdbfe127e1f07fd3fdb1983ef810",
  "inputs": {
    "image_manifest_sha256": "368bdfa0d127480d119c81047fb155ed80e39da392d15b92edc169b1c7ab9e0d",
    "split_manifest_sha256": "fccb4050e4fcdc25d5647d7b6fc7e59662b75e41000f525769a3eb95fac6aa71"
  },
  "git": {
    "commit": "8989077f595c3d14c04bb35d1cba7410b3fa3f0c",
    "dirty": false
  },
  "class_names": {
    "0": "lobster"
  },
  "target_ultralytics_version": "8.4.155",
  "link_mode": "symlink",
  "splits": {
    "train": {
      "images": 1433,
      "positive": 1274,
      "instances": 4000,
      "negative": 159
    },
    "val": {
      "images": 324,
      "negative": 55,
      "instances": 1010,
      "positive": 269
    },
    "test": {
      "images": 348,
      "positive": 321,
      "instances": 1512,
      "negative": 27
    }
  },
  "not_in

In [10]:
# 10. Ultralytics reads the view exactly as built. Must print VERIFIED.
sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", "data/yolo/lobster_v1")


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 127.3±38.7 MB/s, size: 2084.7 KB)

val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 2 images, 2 backgrounds, 0 corrupt: 0% ──────────── 2/1433 5.3it/s 0.1s<4:30
val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 10 images, 10 backgrounds, 0 corrupt: 0% ──────────── 10/1433 23.9it/s 0.2s<59.5s
val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 16 images, 15 backgrounds, 0 corrupt: 1% ──────────── 16/1433 33.6it/s 0.3s<42.1s
val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 23 images, 22 backgrounds, 0 corrupt: 1% ──────────── 23/1433 44.5it/s 0.4s<31.7s
val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 30 images, 27 backgrounds, 0 corrupt: 2% ──────────── 30/1433 51.4it/s 0.5s<27.3s
val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train... 35 images, 31 backgrounds, 0 corrupt: 2% ──────────── 35/1433 49.6it/s 0.7s<28.2s
val: Scanning /content/uwct/data/yolo

In [11]:
# commit check
import subprocess

print("Current commit:")
print(
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd="/content/uwct",
        text=True
    ).strip()
)

print("\nCandidate declaration:")
print(
    open(
        "/content/uwct/experiments/final_candidates.csv",
        encoding="utf-8"
    ).read()
)

Current commit:
8989077f595c3d14c04bb35d1cba7410b3fa3f0c

Candidate declaration:
experiment_id,declared_utc,decision,reason
E001,2026-09-19T19:50:42Z,candidate,"E001 baseline selected for held-out final evaluation after completed validation run"


In [12]:
# 11. Preflight: every training check + load (download) the exact pretrained checkpoint.
#     If rtdetr-l.pt is unavailable in ultralytics 8.4.155 this STOPS -- report it; do not change the model.
TRAIN = [sys.executable, "scripts/train_yolo.py", "--config", EXPERIMENT_CONFIG,
         "--runs-root", RUNS_ROOT, "--weights-dir", WEIGHTS_DIR]
sh(*TRAIN, "--preflight-only")

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2004.5±773.4 MB/s, size: 2004.4 KB)

val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train.cache... 1433 images, 159 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1433/1433 193.9Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2529.2±1473.6 MB/s, size: 2086.9 KB)

val: Scanning /content/uwct/data/yolo/lobster_v1/labels/val.cache... 324 images, 55 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 324/324 64.7Mit/s 0.0s

{
  "kind": "full",
  "created_utc": "2026-10-13T15:47:03+00:00",
  "run_name": "E003_preflight",
  "experiment": {
    "id": "E003",
    "slug": "rtdetr_baseline",
    "parent": "E001",
    "changed": "model yolo26s.pt -> rtdetr-l.pt",
    "config_path": "configs/experiments/E003_rtdetr_baseline.yaml",
    "config_sha256": "1fdc754799bcecdd6586d2f6109d2299df467ef7635ee9e9663cdf38458241bf",
    "config": {
      "id": "E003",
      "slug": "rtdetr_baseline",
      "parent": "E001",
      "changed": "model yolo26s.

In [13]:
# 12. Smoke test: 1 epoch on 10% of train, validated on val -> RUNS_ROOT/E003_smoke_<utc>/
RUN_SMOKE = True  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
sh(*TRAIN, "--smoke")

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1737.0±209.4 MB/s, size: 2017.7 KB)

val: Scanning /content/uwct/data/yolo/lobster_v1/labels/train.cache... 1433 images, 159 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1433/1433 300.5Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3140.1±1208.3 MB/s, size: 2310.5 KB)

val: Scanning /content/uwct/data/yolo/lobster_v1/labels/val.cache... 324 images, 55 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 324/324 84.9Mit/s 0.0s
New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/c

In [14]:
# 12b. Smoke-test check (RT-DETR): losses must be finite before any full run (Ultralytics notes
#      AMP can give NaN for RT-DETR). Reads the newest E003 smoke run's results.csv on Drive.
import csv, glob, math

smoke_dirs = sorted(glob.glob(os.path.join(RUNS_ROOT, EXPERIMENT_ID + "_smoke_*")))
assert smoke_dirs, "no smoke run found -- run cell 12 first"
with open(os.path.join(smoke_dirs[-1], "train", "results.csv"), newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
last = {k.strip(): v for k, v in rows[-1].items()}
print(smoke_dirs[-1])
for k, v in last.items():
    print(f"{k:>28}: {v}")
bad = [k for k, v in last.items() if "loss" in k and not math.isfinite(float(v))]
assert not bad, f"non-finite losses {bad}: STOP -- record a decision before changing batch/amp (D-015)"
print("Smoke losses are finite.")

/content/drive/MyDrive/uwct/runs/E003_smoke_20261013T154708
                       epoch: 1
                        time: 57.5799
             train/giou_loss: 1.31299
              train/cls_loss: 7.44994
               train/l1_loss: 0.52978
        metrics/precision(B): 0.00644
           metrics/recall(B): 0.51287
            metrics/mAP50(B): 0.00395
         metrics/mAP50-95(B): 0.00161
               val/giou_loss: 1.05746
                val/cls_loss: 13.2437
                 val/l1_loss: 0.48938
                      lr/pg0: 0.002
                      lr/pg1: 0.002
                      lr/pg2: 0.002
Smoke losses are finite.


In [15]:
# # 13. Full training run -> RUNS_ROOT/E003/. Only after the smoke test passed and the run is approved.
# RUN_FULL = True  # set to True deliberately
# assert RUN_FULL, "Full training not enabled"
# sh(*TRAIN)

In [ ]:

# 13. Resume the existing E003 training run.
# Use this only if you want to continue the previous experiment.

RUN_FULL = True
assert RUN_FULL, "Full training not enabled"

sh(*TRAIN, "--resume")


New https://pypi.org/project/ultralytics/8.4.177 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/uwct/runs/E003/data_trainval.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

In [ ]:
# 14. Resume after a Colab disconnect: re-run cells 1-10 (same COMMIT), then this cell.
RESUME = True  # set to True deliberately
assert RESUME, "Resume not enabled"
sh(*TRAIN, "--resume")

In [ ]:
# 15. Summary of the E003 run: training progress from results.csv, then the run record.
#     Then download run_record.json and, on the local PC:
#     python scripts/register_run.py <path>/run_record.json
import csv, json, math

RUN_DIR = os.path.join(RUNS_ROOT, EXPERIMENT_ID)
RESULTS_CSV = os.path.join(RUN_DIR, "train", "results.csv")
assert os.path.isfile(RESULTS_CSV), f"no {RESULTS_CSV} -- the full run (cell 13) has not written any epoch yet"
with open(RESULTS_CSV, newline="", encoding="utf-8") as f:
    epochs = [{k.strip(): v.strip() for k, v in r.items()} for r in csv.DictReader(f)]
assert epochs, f"{RESULTS_CSV} has no epoch rows yet"


def _finite(v):
    try:
        return math.isfinite(float(v))
    except ValueError:
        return False


loss_cols = [k for k in epochs[0] if "loss" in k]
nan_epochs = [r["epoch"] for r in epochs if not all(_finite(r[k]) for k in loss_cols)]
show = ["epoch"] + loss_cols + [k for k in epochs[0] if k.startswith("metrics/")]
print(f"{RESULTS_CSV}: {len(epochs)} epoch(s) written")
print(" | ".join(f"{k.split('/')[-1]:>14}" for k in show))
for r in epochs[-5:]:
    print(" | ".join(f"{r[k]:>14}" for k in show))
if nan_epochs:
    print(f"\nSTOP: non-finite losses from epoch {nan_epochs[0]} ({len(nan_epochs)} epoch(s)). "
          "The run diverged; resuming from last.pt cannot recover it. Do not register it or evaluate it on test. "
          "Record a decision in docs/DECISIONS.md before changing batch/amp/lr (D-015).")

RECORD = os.path.join(RUN_DIR, "run_record.json")
if not os.path.isfile(RECORD):
    print(f"\nno {RECORD} yet -- the run is unfinished; it is written when training ends.")
else:
    rec = json.load(open(RECORD, encoding="utf-8"))
    print(json.dumps({k: rec[k] for k in ("run_name", "git", "seed", "ultralytics_version", "val_metrics")}, indent=2))
    print("dataset:", rec["dataset"]["version"], rec["dataset"]["fingerprint"])
    print("best:", rec["outputs"]["best"])

# FINAL TEST EVALUATION

The **only** place the test split is used (CLAUDE.md rule 7). Run it once per pre-declared final
candidate, after its training run is finished, registered and reviewed. The results never feed back
into training, thresholds or model choice.

Before running:
1. Add the experiment to `experiments/final_candidates.csv`, and declare it in `docs/EXPERIMENTS.md`.
2. Commit, push, and set `COMMIT` (cell 1) to that commit.
3. Run cells 1–10 in this session.

Afterwards, log the evaluation in `docs/EXPERIMENTS.md` (Test-evaluation log).

In [ ]:
# 16. FINAL TEST EVALUATION -- the only cell that touches the test split.
#     scripts/final_test_eval.py -> src/training/final_test.py:
#       a. operating confidence chosen on VAL (train/val data file), frozen;
#       b. RTDETR(best.pt).val(data=<full data.yaml incl. test>, split="test", plots=True);
#       c. prediction examples (one test image per test session) at the val-chosen confidence.
#     Everything is written to RUNS_ROOT/<experiment>/final_test/ on Drive (outside Git).
#     Refused unless the experiment is declared in experiments/final_candidates.csv at COMMIT.
import csv, math

FINAL_TEST_EXPERIMENT = EXPERIMENT_ID
RUN_FINAL_TEST = False  # set to True deliberately, once, after the declaration is committed
assert RUN_FINAL_TEST, "Final test evaluation not enabled"

# Readable pre-checks (final_test_eval.py enforces the declaration too; this explains a refusal up front).
with open(os.path.join(REPO_DIR, "experiments", "final_candidates.csv"), newline="", encoding="utf-8") as f:
    declared = {r["experiment_id"] for r in csv.DictReader(f)}
assert FINAL_TEST_EXPERIMENT in declared, (
    f"{FINAL_TEST_EXPERIMENT} is not declared in experiments/final_candidates.csv at COMMIT {COMMIT[:7]} "
    f"(declared: {sorted(declared)}). Declare it (and in docs/EXPERIMENTS.md), commit, push, set COMMIT, rerun cells 1-10.")
_dir = os.path.join(RUNS_ROOT, FINAL_TEST_EXPERIMENT)
assert os.path.isfile(os.path.join(_dir, "run_record.json")), f"no finished run record in {_dir} -- finish training first"
with open(os.path.join(_dir, "train", "results.csv"), newline="", encoding="utf-8") as f:
    _rows = [{k.strip(): v.strip() for k, v in r.items()} for r in csv.DictReader(f)]


def _finite(v):
    try:
        return math.isfinite(float(v))
    except ValueError:
        return False


_bad = [r["epoch"] for r in _rows if not all(_finite(v) for k, v in r.items() if "loss" in k)]
assert not _bad, f"{FINAL_TEST_EXPERIMENT} has non-finite training losses (from epoch {_bad[0]}): not a valid test candidate"

print("=" * 72 + "\nFINAL TEST EVALUATION -- " + FINAL_TEST_EXPERIMENT + "\n" + "=" * 72)
sh(sys.executable, "scripts/final_test_eval.py", "--experiment", FINAL_TEST_EXPERIMENT,
   "--runs-root", RUNS_ROOT, "--confirm-final-test")

In [ ]:
# 17. FINAL TEST EVALUATION -- results and plots, read back from the run folder on Drive.
import glob
import json
from IPython.display import Image, Markdown, display

FINAL_DIR = os.path.join(RUNS_ROOT, FINAL_TEST_EXPERIMENT, "final_test")
assert os.path.isfile(os.path.join(FINAL_DIR, "final_test_record.json")), (
    f"no final_test_record.json in {FINAL_DIR} -- run cell 16 first")
rec = json.load(open(os.path.join(FINAL_DIR, "final_test_record.json"), encoding="utf-8"))
rep, op = rec["metrics"]["ultralytics_reported"], rec["metrics"]["at_val_conf"]
display(Markdown(f"# FINAL TEST EVALUATION — {rec['experiment_id']} (split = `{rec['split']}`, "
                 f"attempt {rec['attempts_including_this_one']})"))
rows = ["| metric | value |", "|---|---|",
        f"| mAP50 | {rep['map50']:.4f} |", f"| mAP50-95 | {rep['map50_95']:.4f} |"]
if op:
    rows += [f"| precision @ val-chosen conf {op['conf']:.3f} | {op['precision']:.4f} |",
             f"| recall @ val-chosen conf {op['conf']:.3f} | {op['recall']:.4f} |",
             f"| F1 @ val-chosen conf {op['conf']:.3f} | {op['f1']:.4f} |"]
rows += [f"| precision (Ultralytics: max-F1 conf on test) | {rep['precision']:.4f} |",
         f"| recall (Ultralytics: max-F1 conf on test) | {rep['recall']:.4f} |"]
display(Markdown("\n".join(rows)))
print("confusion matrix (rows = predicted, cols = true; lobster, background):",
      (rec["metrics"]["confusion_matrix"] or {}).get("matrix"))

PLOTS = [("Confusion matrix", ["confusion_matrix.png", "confusion_matrix_normalized.png"]),
         ("Precision-recall curve", ["BoxPR_curve.png", "PR_curve.png"]),
         ("F1 curve", ["BoxF1_curve.png", "F1_curve.png"])]
for title, names in PLOTS:
    found = [os.path.join(FINAL_DIR, "test", n) for n in names if os.path.isfile(os.path.join(FINAL_DIR, "test", n))]
    display(Markdown(f"### {title}" + ("" if found else " — not produced by this Ultralytics version")))
    for p in found:
        display(Image(filename=p, width=700))

display(Markdown(f"### Prediction examples at the val-chosen confidence ({op['conf']:.3f})" if op else
                 "### Prediction examples"))
for p in sorted(glob.glob(os.path.join(FINAL_DIR, "predictions", "*.*"))):
    display(Image(filename=p, width=900))
display(Markdown("### Ultralytics validation batches (labels vs predictions with confidence)"))
for p in sorted(glob.glob(os.path.join(FINAL_DIR, "test", "val_batch*_pred.jpg"))):
    display(Image(filename=p, width=900))
display(Markdown("**Limitations:**\n" + "\n".join(f"- {l}" for l in rec["limitations"])))